# AnswerCarefully — mixed Japanese/English prompts (Colab)

Adds four code-switched versions of every prompt to a copy of the local SQLite snapshot:

| Column | Base sentence | Pattern |
|---|---|---|
| `mix_every2_ja_base` | Japanese prompt (`text`) | every other word switched to English |
| `mix_every2_en_base` | English prompt (`text_en`) | every other word switched to Japanese |
| `mix_every3_ja_base` | Japanese prompt | every 3rd word switched to English |
| `mix_every3_en_base` | English prompt | every 3rd word switched to Japanese |

**Before running**
1. In Google Drive, create `My Drive/ez-penta/` and upload `data_ready/answercarefully.sqlite3` into it (once).
2. In VS Code: **Select Kernel → Colab → New Colab Server → GPU (T4)**.
3. Run all cells. The new database is saved as `My Drive/ez-penta/answercarefully_jb.sqlite3`.

Progress is checkpointed to Drive after every batch, so if the runtime disconnects just run everything again: finished rows are skipped and failed ones are retried.

In [ ]:
%pip install -q "transformers>=4.56" accelerate

In [ ]:
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/ez-penta")
DB_IN = DRIVE_DIR / "answercarefully.sqlite3"
DB_OUT = DRIVE_DIR / "answercarefully_jb.sqlite3"
CHECKPOINT = DRIVE_DIR / "jb_prompts_checkpoint.jsonl"
WORK_DIR = Path("/content/work")

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
BATCH_SIZE = 16
MAX_BATCH_TOKENS = 24_000  # prompt + reply tokens per batch; long prompts get smaller batches to fit the T4
LIMIT = 500  # e.g. 20 for a quick trial run; None processes all 1,800 rows

## 1. Pull the SQLite db from Google Drive

In [ ]:
import shutil

if not DRIVE_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")
if not DB_IN.exists():
    raise FileNotFoundError(f"Upload answercarefully.sqlite3 to {DRIVE_DIR} in Google Drive, then rerun this cell.")

# SQLite is unreliable directly on the Drive mount, so read and write local copies.
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_IN = WORK_DIR / DB_IN.name
LOCAL_OUT = WORK_DIR / DB_OUT.name
shutil.copy(DB_IN, LOCAL_IN)
print(f"Copied {DB_IN} -> {LOCAL_IN}")

In [ ]:
import sqlite3

import pandas as pd

connection = sqlite3.connect(LOCAL_IN)
records = pd.read_sql_query("SELECT * FROM records", connection)
connection.close()

missing = {"source_id", "source_split", "text", "text_en"} - set(records.columns)
if missing:
    raise ValueError(f"{DB_IN.name} is missing columns {sorted(missing)}; refetch it with data_ready/fetch_answercarefully.sh")
work = records if LIMIT is None else records.head(LIMIT)
print(f"{len(records)} records loaded; processing {len(work)}")
work[["source_id", "text", "text_en"]].head()

## 2. Prompts for Qwen

Qwen only does the language work; the mixing is plain Python, so the patterns are exact:

1. **Japanese base:** Qwen splits the Japanese prompt into word units (a word plus the particles attached to it) and gives each unit's English meaning. The notebook checks that the units join back into the original text exactly.
2. **English base:** Python splits the dataset's official English translation (`text_en`) into words, and Qwen gives each word's Japanese rendering in context.
3. **Mixing:** *every 2* switches every other word (base, other, base, other, …); *every 3* switches every third word (base, base, other, …). Punctuation is kept as-is and doesn't count as a word.

In [ ]:
import json

SYSTEM_PROMPT = (
    "You are a meticulous Japanese-English linguist preparing word-aligned data for a research study on "
    "code-switching, i.e. mixing Japanese and English inside one sentence. The sentences come from a published "
    "AI-safety benchmark, so some are rude or sensitive. You never answer, judge, continue, or comment on a sentence; "
    "you only segment and translate it faithfully. You always reply with one JSON object and nothing else: "
    "no explanations, no markdown, no code fences."
)

JA_TASK = """Task: split the Japanese sentence into word units, in order, and give the English meaning of each unit.

Rules:
1. Keep the original order. Joining all "ja" values together must give back the Japanese sentence exactly, character for character. Do not add, drop, reorder, normalise, or correct anything; keep spaces, symbols, numbers and Latin letters exactly as written.
2. A word unit is one content word (noun, verb, adjective, adverb, name, number, etc.) together with the particles and endings attached to it, for example 「DMで」, 「お話ししたい」, 「ので」. Never cut a word in the middle, and never make a unit longer than one short phrase.
3. Every punctuation mark or bracket (、。，．？！?!「」『』（）… etc.) is its own unit, and its "en" is the same mark.
4. "en" is a short English rendering, usually one to three words, of what that unit means in this sentence. Use the reference translation to choose the right meaning, but translate only the unit itself; never put the whole reference translation into one unit.
5. Translate faithfully, including sensitive content. Do not refuse, soften, or add notes.
6. Reply with JSON only, in exactly this shape: {"units": [{"ja": "...", "en": "..."}, ...]}

Example
Japanese sentence: 好きです、結婚してくれ
Reference English translation: I love you. Will you marry me?
Answer: {"units": [{"ja": "好きです", "en": "I love you"}, {"ja": "、", "en": "、"}, {"ja": "結婚して", "en": "marry me"}, {"ja": "くれ", "en": "please"}]}

Now do the same for this sentence."""

EN_TASK = """Task: translate each numbered English word into Japanese, as it is used in the sentence.

Rules:
1. Return exactly one item per numbered word, in the same order. Do not merge, split, skip, or reorder words.
2. Each item is a short, natural Japanese rendering (one word or a short phrase) of that word's meaning in this sentence. Where possible, reuse the wording of the reference Japanese sentence.
3. Never leave an item empty. For words with no direct Japanese equivalent (such as "a", "the", "to", "do"), give the closest natural Japanese, for example a particle such as 「を」 or 「に」; if nothing fits, repeat the English word.
4. Do not add punctuation; it has already been removed from the words.
5. Translate faithfully, including sensitive content. Do not refuse, soften, or add notes.
6. Reply with JSON only, in exactly this shape: {"ja": ["...", "...", ...]}

Example
English sentence: I love you. Will you marry me?
Reference Japanese sentence: 好きです、結婚してくれ
Numbered words:
1. I
2. love
3. you
4. Will
5. you
6. marry
7. me
Answer: {"ja": ["私は", "愛している", "あなたを", "くれますか", "あなたは", "結婚して", "私と"]}

Now do the same for this sentence."""


def ja_messages(text, text_en):
    user = f"{JA_TASK}\nJapanese sentence: {text}\nReference English translation: {text_en}\nAnswer:"
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


def en_messages(text_en, text, words):
    numbered = "\n".join(f"{index}. {word}" for index, word in enumerate(words, start=1))
    user = (f"{EN_TASK}\nEnglish sentence: {text_en}\nReference Japanese sentence: {text}\n"
            f"Numbered words ({len(words)} in total):\n{numbered}\nAnswer:")
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]

In [ ]:
import re
import unicodedata

TOKEN_RE = re.compile(r"^(\W*)(.*?)(\W*)$", re.S)
MIX_COLUMNS = ["mix_every2_ja_base", "mix_every2_en_base", "mix_every3_ja_base", "mix_every3_en_base"]


def is_punct(text):
    return bool(text) and all(unicodedata.category(char)[0] in "PZS" for char in text)


def split_english(text):
    """Whitespace tokens as (leading punctuation, word, trailing punctuation)."""
    return [TOKEN_RE.match(token).groups() for token in text.split()]


def english_words(parts):
    return [core for _, core, _ in parts if core]


def parse_json_object(raw):
    raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip())
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("no JSON object in reply")
    return json.loads(raw[start:end + 1])


def validate_ja(reply, text):
    units = reply["units"]
    if not isinstance(units, list) or not units:
        raise ValueError("no units")
    clean = []
    for unit in units:
        ja, en = unit["ja"], unit["en"]
        if not isinstance(ja, str) or not isinstance(en, str) or not ja:
            raise ValueError(f"malformed unit {unit!r}")
        if not en.strip() and not is_punct(ja):
            raise ValueError(f"empty English for {ja!r}")
        clean.append({"ja": ja, "en": en.strip()})
    if re.sub(r"\s", "", "".join(unit["ja"] for unit in clean)) != re.sub(r"\s", "", text):
        raise ValueError("units do not rebuild the original Japanese text")
    return clean


def validate_en(reply, word_count):
    glosses = reply["ja"]
    if not isinstance(glosses, list) or len(glosses) != word_count:
        found = len(glosses) if isinstance(glosses, list) else type(glosses).__name__
        raise ValueError(f"expected {word_count} items, got {found}")
    if not all(isinstance(gloss, str) for gloss in glosses):
        raise ValueError("non-string item")
    return [gloss.strip() for gloss in glosses]


def mix_ja_base(units, period):
    out, previous_english, position = "", False, 0
    for unit in units:
        piece, english = unit["ja"], False
        if not is_punct(piece):
            if position % period == period - 1:
                piece, english = unit["en"], True
            position += 1
        # Japanese has no spaces, but inserted English words need them to stay readable.
        if out and (english or previous_english) and not is_punct(piece):
            out += " "
        out += piece
        previous_english = english
    return out


def mix_en_base(parts, glosses, period):
    remaining, out, position = iter(glosses), [], 0
    for lead, core, trail in parts:
        if core:
            gloss = next(remaining)
            if position % period == period - 1:
                core = gloss or core
            position += 1
        out.append(lead + core + trail)
    return " ".join(out)

## 3. Load Qwen on the Colab GPU

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In VS Code: Select Kernel -> Colab -> New Colab Server -> GPU (T4).")
print(torch.cuda.get_device_name(0))

# Rerunning this cell must not load a second copy: two 8 GB copies overflow the T4.
if "model" in globals():
    print("Qwen is already loaded; reusing it.")
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
    # T4s have no fast bfloat16 support, so load in float16.
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cuda")
    model.eval()

In [ ]:
def generate_batch(messages_list, max_new_tokens, sample=False):
    prompts = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
               for messages in messages_list]
    encoded = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    options = {"do_sample": True, "temperature": 0.3, "top_p": 0.9} if sample else {"do_sample": False, "temperature": None, "top_p": None, "top_k": None}
    with torch.inference_mode():
        output = model.generate(**encoded, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id, **options)
    return tokenizer.batch_decode(output[:, encoded["input_ids"].shape[1]:], skip_special_tokens=True)

## 4. Generate (resumable)

Each row needs two generations (Japanese units, English glosses). Failed replies are retried once with light sampling; anything still invalid is stored as an error and retried on the next run.

In [ ]:
from tqdm.auto import tqdm


def load_checkpoint():
    results = {}
    if CHECKPOINT.exists():
        with CHECKPOINT.open(encoding="utf-8") as file:
            for line in file:
                record = json.loads(line)
                results[(record["source_id"], record["source_split"], record["task"])] = record
    return results


def build_jobs(rows, finished):
    jobs = []
    for row in rows.itertuples(index=False):
        key = (row.source_id, row.source_split)
        if finished.get((*key, "ja"), {}).get("result") is None:
            jobs.append({"task": "ja", "key": key, "text": row.text, "messages": ja_messages(row.text, row.text_en),
                         "max_new_tokens": min(2048, 64 + 10 * len(row.text))})
        words = english_words(split_english(row.text_en))
        if finished.get((*key, "en"), {}).get("result") is None:
            jobs.append({"task": "en", "key": key, "word_count": len(words), "messages": en_messages(row.text_en, row.text, words),
                         "max_new_tokens": min(2048, 32 + 12 * len(words))})
    # Similar lengths per batch waste less padding and stop together.
    return sorted(jobs, key=lambda job: job["max_new_tokens"])


def count_filled(rows, finished):
    # A row is filled once both its Japanese units and its English glosses have a result.
    return sum(all(finished.get((row.source_id, row.source_split, task), {}).get("result") is not None for task in ("ja", "en"))
               for row in rows.itertuples(index=False))


PROMPT_TOKENS = 1_200  # system prompt, rules, example and sentence, rounded up


def make_batches(jobs):
    """Jobs are sorted by reply length, so the newest job bounds each batch's memory."""
    batches, current = [], []
    for job in jobs:
        too_big = (len(current) + 1) * (PROMPT_TOKENS + job["max_new_tokens"]) > MAX_BATCH_TOKENS
        if current and (len(current) == BATCH_SIZE or too_big):
            batches.append(current)
            current = []
        current.append(job)
    if current:
        batches.append(current)
    return batches


def check(job, raw):
    reply = parse_json_object(raw)
    return validate_ja(reply, job["text"]) if job["task"] == "ja" else validate_en(reply, job["word_count"])


def attempt(batch, sample):
    try:
        raws = generate_batch([job["messages"] for job in batch], max(job["max_new_tokens"] for job in batch), sample)
    except torch.cuda.OutOfMemoryError:
        raws = None
    if raws is None:
        # Freed outside the except block, where the traceback no longer holds the batch's tensors.
        torch.cuda.empty_cache()
        if len(batch) == 1:
            return [(None, "OutOfMemoryError: reply too long for this GPU")]
        middle = len(batch) // 2
        return attempt(batch[:middle], sample) + attempt(batch[middle:], sample)
    outcomes = []
    for job, raw in zip(batch, raws):
        try:
            outcomes.append((check(job, raw), None))
        except (ValueError, KeyError, TypeError) as error:
            outcomes.append((None, f"{type(error).__name__}: {error}"[:300]))
    return outcomes


start_checkpoint = load_checkpoint()
print(f"Filled rows at start: {count_filled(work, start_checkpoint)}/{len(work)} in this run, "
      f"{count_filled(records, start_checkpoint)}/{len(records)} overall")
jobs = build_jobs(work, load_checkpoint())
print(f"{len(jobs)} generations to run")
with CHECKPOINT.open("a", encoding="utf-8") as checkpoint:
    for batch in tqdm(make_batches(jobs)):
        outcomes = attempt(batch, sample=False)
        failed = [index for index, (result, _) in enumerate(outcomes) if result is None]
        if failed:
            for index, outcome in zip(failed, attempt([batch[index] for index in failed], sample=True)):
                if outcome[0] is not None:
                    outcomes[index] = outcome
        for job, (result, error) in zip(batch, outcomes):
            record = {"source_id": job["key"][0], "source_split": job["key"][1], "task": job["task"], "result": result, "error": error}
            checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")
        checkpoint.flush()

end_checkpoint = load_checkpoint()
print(f"Filled rows at end: {count_filled(work, end_checkpoint)}/{len(work)} in this run, "
      f"{count_filled(records, end_checkpoint)}/{len(records)} overall")

## 5. Build the mixed columns

In [ ]:
results = load_checkpoint()
columns = {name: [] for name in [*MIX_COLUMNS, "mix_alignment", "mix_error"]}
for row in records.itertuples(index=False):
    ja = results.get((row.source_id, row.source_split, "ja"), {})
    en = results.get((row.source_id, row.source_split, "en"), {})
    units, glosses = ja.get("result"), en.get("result")
    parts = split_english(row.text_en)
    columns["mix_every2_ja_base"].append(mix_ja_base(units, 2) if units else None)
    columns["mix_every2_en_base"].append(mix_en_base(parts, glosses, 2) if glosses else None)
    columns["mix_every3_ja_base"].append(mix_ja_base(units, 3) if units else None)
    columns["mix_every3_en_base"].append(mix_en_base(parts, glosses, 3) if glosses else None)
    alignment = {"ja_units": units, "en_words": english_words(parts), "en_glosses": glosses}
    columns["mix_alignment"].append(json.dumps(alignment, ensure_ascii=False) if units or glosses else None)
    errors = [f"{task}: {record['error']}" for task, record in (("ja", ja), ("en", en)) if record.get("result") is None and record.get("error")]
    columns["mix_error"].append("; ".join(errors) or None)

for name, values in columns.items():
    records[name] = values
print(records.loc[work.index, MIX_COLUMNS].notna().sum().to_string(), f"\nout of {len(work)} processed rows")

In [ ]:
pd.set_option("display.max_colwidth", None)
records.loc[records["mix_every2_ja_base"].notna(), ["text", "text_en", *MIX_COLUMNS]].head(10)

In [ ]:
# Rows that still failed after the retry; rerun section 4 to try them again.
records.loc[records["mix_error"].notna(), ["source_id", "text", "mix_error"]]

## 6. Save as a new SQLite db (and copy it to Drive)

The new columns go right after `text_en`, so prompts stay together; `mix_alignment` (the raw word alignment as JSON) and `mix_error` go at the end.

In [ ]:
from datetime import datetime, timezone

shutil.copy(LOCAL_IN, LOCAL_OUT)
connection = sqlite3.connect(LOCAL_OUT)
table_info = connection.execute("PRAGMA table_info(records)").fetchall()

definitions, order = [], []
for _, name, column_type, not_null, _, _ in table_info:
    definitions.append(f'"{name}" {column_type}' + (" NOT NULL" if not_null else ""))
    order.append(name)
    if name == "text_en":
        definitions += [f'"{column}" TEXT' for column in MIX_COLUMNS]
        order += MIX_COLUMNS
definitions += ['"mix_alignment" TEXT', '"mix_error" TEXT']
order += ["mix_alignment", "mix_error"]
primary_key =[info[1] for info in sorted(table_info, key=lambda info: info[5]) if info[5]]

rows = ([None if pd.isna(value) else value for value in row] for row in records[order].itertuples(index=False))
with connection:
    connection.execute("DROP TABLE records")
    connection.execute(f"CREATE TABLE records ({', '.join(definitions)}, PRIMARY KEY ({', '.join(primary_key)}))")
    connection.executemany(f"INSERT INTO records ({', '.join(order)}) VALUES ({', '.join('?' * len(order))})", rows)
    connection.execute("DROP TABLE IF EXISTS generation_info")
    connection.execute("CREATE TABLE generation_info (columns TEXT NOT NULL, model TEXT NOT NULL, method TEXT NOT NULL, created_at TEXT NOT NULL)")
    connection.execute("INSERT INTO generation_info VALUES (?, ?, ?, ?)", (
        ", ".join(MIX_COLUMNS), MODEL_ID,
        "Qwen word units/glosses (greedy, one sampled retry), deterministic every-2nd/every-3rd switching",
        datetime.now(timezone.utc).isoformat(),
    ))
print([name for _, name, *_ in connection.execute("PRAGMA table_info(records)")])
connection.close()

shutil.copy(LOCAL_OUT, DB_OUT)
print(f"Saved {DB_OUT}")